In [1]:
import numpy as np
import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs
from pathlib import Path
plt.rcParams["font.size"] = 16

un panel avec des courbes <x>(t) pour different l dans le cas périodique pour le cas mu,theta=160,20

un panel avec des courbes <x>(t) pour different l dans le cas aléatoire pour le cas mu,theta=160,20

un panel avec des courbes <x>(t) pour different bpmin dans le cas aléatoire pour le cas mu,theta=160,20

un panel avec des courbes v_init vs mu pour différent l dans le cas périodique pour le cas theta=20

un panel avec des courbes v_init vs mu pour différent l dans le cas aléatoire pour le cas theta=20

un panel avec des courbes v_init vs mu pour différent bpmin dans le cas aléatoire pour le cas theta=20

In [26]:
root  = Path("/home/nicolas/Documents/Workspace/nucleo/outputs/NUCLEO__PSMN__2026-06-19")
paths = [str(p) for p in root.glob("nucleo__*")]

theta = 20
s_selected = 150
ls = np.array([5, 10, 20, 50, 100, 150])
bpmin = 0
ylims = [-50, 700]

ARRAY_COLS = []

df_data = (
    pl.scan_parquet(paths)
    .filter(
        (pl.col("theta") == theta) &
        (pl.col("s")     == s_selected) &
        (pl.col("bpmin") == 0) &
        (pl.col("l").is_in(ls))
    )
    .select(cs.numeric() | cs.boolean() | cs.string() | cs.by_name(*ARRAY_COLS))
    .collect()
    .sort(["land", "bpmin", "l", "mu"])
)

print(df_data)
print(df_data.columns)

shape: (1_515, 61)
┌──────┬───────┬──────┬───────┬───┬───────┬───────────┬───────────┬───────┐
│ algo ┆ fact  ┆ mode ┆ dstr  ┆ … ┆ dx_mp ┆ dt_mean   ┆ dt_med    ┆ dt_mp │
│ ---  ┆ ---   ┆ ---  ┆ ---   ┆   ┆ ---   ┆ ---       ┆ ---       ┆ ---   │
│ str  ┆ bool  ┆ str  ┆ bool  ┆   ┆ f64   ┆ f64       ┆ f64       ┆ f64   │
╞══════╪═══════╪══════╪═══════╪═══╪═══════╪═══════════╪═══════════╪═══════╡
│ 1S   ┆ false ┆ none ┆ false ┆ … ┆ 95.5  ┆ 15.928331 ┆ 11.044052 ┆ 0.5   │
│ 1S   ┆ false ┆ none ┆ false ┆ … ┆ 99.5  ┆ 15.89962  ┆ 10.986653 ┆ 0.5   │
│ 1S   ┆ false ┆ none ┆ false ┆ … ┆ 107.5 ┆ 15.996025 ┆ 11.03436  ┆ 0.5   │
│ 1S   ┆ false ┆ none ┆ false ┆ … ┆ 110.5 ┆ 16.02246  ┆ 11.077971 ┆ 0.5   │
│ 1S   ┆ false ┆ none ┆ false ┆ … ┆ 118.5 ┆ 16.023551 ┆ 11.120506 ┆ 0.5   │
│ …    ┆ …     ┆ …    ┆ …     ┆ … ┆ …     ┆ …         ┆ …         ┆ …     │
│ 1S   ┆ false ┆ none ┆ false ┆ … ┆ 577.5 ┆ 7.3106e8  ┆ 2.110603  ┆ 0.5   │
│ 1S   ┆ false ┆ none ┆ false ┆ … ┆ 586.5 ┆ 1.7001e9  ┆ 2.101813  ┆ 0

In [32]:
df_periodic = df_data.filter(pl.col("land") == "periodic")

groups = (
    df_periodic
    .group_by("l")
    .agg([
        pl.col("mu"),
        pl.col("vf")
    ])
    .sort("l")
)


plt.figure(figsize=(8, 6), dpi=1200)

for row in groups.iter_rows(named=True):
    l = row["l"]

    mu_vals = np.array(row["mu"])
    vf_vals = np.array(row["vf"])

    # tri important si pas garanti
    idx = np.argsort(mu_vals)
    mu_vals = mu_vals[idx]
    vf_vals = vf_vals[idx]

    plt.plot(mu_vals, vf_vals, marker="s", label=rf"$l={l}$")

plt.grid(True, alpha=0.3)
plt.xlabel(r"$\mu$")
plt.ylabel(r"$v_{init}$")
plt.title("Periodic")
plt.ylim(ylims)
plt.legend()
plt.tight_layout()
plt.show()

In [31]:
df_random = df_data.filter(pl.col("land") == "random")

groups = (
    df_random
    .group_by("l")
    .agg([
        pl.col("mu"),
        pl.col("vf")
    ])
    .sort("l")
)


plt.figure(figsize=(8, 6), dpi=1200)

for row in groups.iter_rows(named=True):
    l = row["l"]

    mu_vals = np.array(row["mu"])
    vf_vals = np.array(row["vf"])

    # tri important si pas garanti
    idx = np.argsort(mu_vals)
    mu_vals = mu_vals[idx]
    vf_vals = vf_vals[idx]

    plt.plot(mu_vals, vf_vals, marker="o", label=rf"$l={l}$")

plt.grid(True, alpha=0.3)
plt.xlabel(r"$\mu$")
plt.ylabel(r"$v_{init}$")
plt.title("Random")
plt.ylim(ylims)
plt.legend()
plt.tight_layout()
plt.show()

In [30]:
df_homogen = df_data.filter(pl.col("land") == "periodic")

ls = np.sort(df_homogen["l"].unique().to_numpy())
mu_vals = np.sort(df_homogen["mu"].unique().to_numpy())

plt.figure(figsize=(8, 6), dpi=1200)

for l in ls:
    vf_vals = mu_vals * (l / (s_selected + l))

    plt.plot(mu_vals, vf_vals, marker="^", label=rf"$l={l}$")

plt.grid(True, alpha=0.3)
plt.xlabel(r"$\mu$")
plt.ylabel(r"$v_{init}$")
plt.title("Homogen")
plt.ylim(ylims)
plt.legend()
plt.tight_layout()
plt.show()

# .